# Binh - Chap 10/11 Exercises
10.1.6d, 10.2.2c, 10.4.2a, 11.2.4b, 11.3.4b

In [3]:
# setup
import numpy as np
from numpy import linalg as la
import sympy as sp

## 10.1.6d:
We are asked to use fixed-point iteration to find a solution to the nonlinear system to within $10^{-5}$ in the infinity norm. This means that each of the 3 equations in the system has to be converted into a fixed point problem:


In [6]:
# first, we declare g(x) with candidate fixed point functions
def g(x):
    x1, x2, x3 = x
    return np.array([
        np.sqrt(x2 + 2*x3 - 2*x2**2),
        np.sqrt((x1**2 + 10*x3)/8),
        x1**2/(7*x2)
    ])

# code for iteration implementation
def iter(fn,st,n):
    pts = []
    x = st
    for i in range(0,n):
        pts.append(x)
        x = fn(x)
    return pts

# choosing a reasonable starting point (not zeros)
a = iter(g, (0.1,0.1,0.1), 63)
print(a)

# since we have no idea what the true solution is, we test the inf norm difference between the iterations
la.norm(a[-1] - a[-2], ord = np.inf)

# after about 60 iterations, the norm of the difference between 2 iterates has dropped below the threshold

[(0.1, 0.1, 0.1), array([0.52915026, 0.35531676, 0.01428571]), array([0.36247509, 0.22990681, 0.11257561]), array([0.59105308, 0.3964127 , 0.08164065]), array([0.49538665, 0.38173129, 0.12589468]), array([0.58487869, 0.43364079, 0.0918402 ]), array([0.49115426, 0.39693908, 0.11269468]), array([0.55426274, 0.41354857, 0.08681883]), array([0.49511756, 0.38330724, 0.10612234]), array([0.54927501, 0.4040985 , 0.09136326]), array([0.51013119, 0.38976526, 0.10665824]), array([0.54703548, 0.40724934, 0.09538116]), array([0.51605   , 0.39576814, 0.10497178]), array([0.54078356, 0.40558991, 0.09612685]), array([0.51849518, 0.39587172, 0.10300582]), array([0.53707963, 0.4029416 , 0.09701456]), array([0.5217728 , 0.39664218, 0.1022674 ]), array([0.53528211, 0.40232463, 0.09805414]), array([0.52412088, 0.3979743 , 0.10173978]), array([0.53356047, 0.40188626, 0.09860748]), array([0.52542943, 0.39855388, 0.10119664]), array([0.53221873, 0.40125467, 0.09895636]), array([0.5264568 , 0.3988766 , 0.1008

np.float64(7.540649228476859e-06)

## 10.2.2c
Using sympy's capacities, we can construct the Jacobian and implement Newton's method without much difficulty:

In [4]:
# first, declare sympy vars and provided matrix (switching the const around so F(x) = 0)
x1, x2, x3 = sp.symbols('x1 x2 x3')
f = sp.Matrix([15*x1 + x2**2 - 4*x3 - 13, x1**2 + 10*x2 - x3 - 11, x2**3 - 25*x3 + 22])

# then, the provided newton iteration
def newton(f,vars,st,n):
    x = np.array(st)
    J = f.jacobian(vars)
    for _ in range(0,n):
        sublis = list(zip(vars,x))
        A = np.array(J.subs(sublis)).astype(np.float64)
        y = np.array(f.subs(sublis)).astype(np.float64)
        z = np.linalg.solve(A,y)
        x = x - np.transpose(z)[0]
        print(x)
    return x

# the call:
a = newton(f, sp.Matrix([x1, x2, x3]), [0,0,0], 2)

[1.10133333 1.188      0.88      ]
[1.03668708 1.08592383 0.92977932]


## 10.4.2a
We are tasked to solve the same nonlinear system using the steepest descent technique. We convert the function appropriately into $g(x)$, then implement the algorithm written out in the text.

In [18]:
# declaring the same system again:
x1, x2, x3 = sp.symbols('x1 x2 x3')
f = sp.Matrix([15*x1 + x2**2 - 4*x3 - 13, x1**2 + 10*x2 - x3 - 11, x2**3 - 25*x3 + 22])
g = sum([a**2 for a in f])

def steepdesc(vars, st, TOL, N, g):
    x = np.array(st)
    # symbolic substitution to calculate grad
    grad = sp.Matrix([sp.diff(g, v) for v in vars])
    for k in range(1, N+1):
        sublis = list(zip(vars,x))
        g1 = float(g.subs(sublis))
        z = np.array(grad.subs(sublis)).astype(np.float64).flatten()
        z0 = la.norm(z, ord=2)
        # 0 gradient, no improvement, return current point
        if z0 == 0:
            print('0 gradient')
            print(x, g1)
            return(x)
        z = z/z0
        alpha1 = 0
        alpha3 = 1
        g3 = float(g.subs(list(zip(vars, x-alpha3*z))))
        while g3 >= g1:
            alpha3 = alpha3/2
            g3 = float(g.subs(list(zip(vars, x-alpha3*z))))

            if alpha3 < TOL/2:
                print('no likely improvement')
                print(x, g1)
                return x

        alpha2 = alpha3/2
        g2 = float(g.subs(list(zip(vars, x-alpha2*z))))
        h1 = (g2-g1)/alpha2
        h2 = (g3-g2)/(alpha3-alpha2)
        h3 = (h2-h1)/alpha3
        alpha0 = 0.5*(alpha2 - (h1/h3))
        g0 = float(g.subs(list(zip(vars, x-alpha0*z))))
        # choosing between the different values
        if g0 < g3:
            alpha = alpha0
            gnew = g0
        else:
            alpha = alpha3
            gnew = g3
        x = x - alpha * z
        if np.abs(gnew - g1) < TOL:
            print(x, gnew)
            return x
    print('max iterations reached, no cigar')
    return x    

ans = steepdesc([x1, x2, x3], [0,0,0], 0.05, 13, g)     

no likely improvement
[1.04888334 1.04794589 0.92014291] 0.16040948577733177


## 11.2.4b
We are given a boundary value problem we will need to tackle with the nonlinear shooting method. For the iteration of t, I will be using the Newton method.

In [ ]:
# first, define a runge-kutta solver for the final solution
def runge_kutta(f,t0,y0,h,n):
    y = y0
    t = t0
    pts = [(t,y)]
    for _ in range(0,n):
        k1 = f(t,y)
        k2 = f(t+h/2,y+k1*h/2)
        k3 = f(t+h/2,y+k2*h/2)
        k4 = f(t+h,y+k3*h)
        y = y + (k1+2*k2+2*k3+k4)*h/6
        t = t + h
        pts.append((t,y))
    return pts

# then, we define the function:
def shooting(f, fy, fyp, a, b, alpha, beta, N, TOL, M, TK=None):
    # where fy and fyp are partials wrt y and y' to calculate u2'
    if TK is None:
        TK = (beta-alpha)/(b-a)
    # h = 0.1 already
    h = (b-a)/N
    # initiate k before the while loop
    k = 1 
    # then the system of w1, w2, u1, u2:
    def rhs(x, Y):
        w1, w2, u1, u2 = Y
        return(
            np.array([w2,
                    f(x,w1,w2),
                    u2,
                    fy(x, w1, w2)*u1 + fyp(x, w1, w2)*u2])
        )
    
    while k <= M:
        # starting val of y, y', predefined z and z'
        Y0 = np.array([alpha, TK, 0, 1])
        pts = runge_kutta(rhs, a, Y0, h, N)
        # this gets us to estimates of the point on b's end given t = TK
        # since points are returned in (x,[w1, w2, u1, u2]) form:
        YN = pts[-1][1][0]
        ZN = pts[-1][1][2]
        if np.abs(YN - beta) <= TOL:
            return pts # since the solver doesnt have any correction factor
        
        TK = TK - (YN-beta)/ZN
        k = k+1
    print("Max iter reached")
    return pts

# now, define the equation we're supposed to solve:
def f(x, y, yp):
    return 2*y**3 - 6*y - 2*x**3

def fy(x,y,yp):
    return 6*y**2 - 6

def fyp(x,y,yp):
    return 0 # since there is no dependence on y' in the equation

# calling the function 
pts = shooting(f, fy, fyp, 1, 2, 2, 5/2, 10, 1e-4, 20)
est = np.array([(pt[0], pt[1][0]) for pt in pts])
print("The shooting method nets us these approximations\n", est)

def soln(x): # getting the true answer for y
    return np.array([x, x + 1/x]).T

x = np.linspace(1, 2, 11)
true = soln(x)

# now to get an error vector:
print("The errors we are making at each step are", est[:,1]-true[:,1])

The shooting method nets us these approximations
 [[1.         2.        ]
 [1.1        2.00910296]
 [1.2        2.03335064]
 [1.3        2.06924973]
 [1.4        2.11430448]
 [1.5        2.16668438]
 [1.6        2.22501639]
 [1.7        2.28825041]
 [1.8        2.35556967]
 [1.9        2.42632936]
 [2.         2.50001376]]
The errors we are making at each step are [0.00000000e+00 1.20462719e-05 1.73080219e-05 1.89588246e-05
 1.87631017e-05 1.77181725e-05 1.63925790e-05 1.51187842e-05
 1.41144668e-05 1.35732474e-05 1.37552835e-05]


## 11.3.4b
We are asked to solve a system using the linear finite difference method.

In [60]:
# first, we convert the thing into the right form:
# in this case, the problem is y'' = 0*y' - 4y + cos(x)
def q(x):
    return -4

def r(x):
    return np.cos(x)

def p(x):
    return 0

def linfin(p, q, r, a, b, alpha, beta, N):
    # we need to code the vectors differently since a b is used
    h = (b-a)/(N+1)
    # init the vectors:
    A = np.zeros(N+2)
    B = np.zeros(N+2)
    C = np.zeros(N+2)
    D = np.zeros(N+2)

    l = np.zeros(N+2)
    u = np.zeros(N+2)
    z = np.zeros(N+2)
    w = np.zeros(N+2)

    # first values
    x = a + h
    A[1] = 2 + h**2*q(x)
    B[1] = -1 + (h/2)*p(x)
    D[1] = -(h**2)*r(x) + (1 + (h/2)*p(x))*alpha

    # loop to N-1
    for i in range(2, N):
        x = a + i*h
        A[i] = 2 + h**2*q(x)
        B[i] = -1 + (h/2)*p(x)
        C[i] = -1 - (h/2)*p(x)
        D[i] = -h**2*r(x)
    
    # step N
    x = b - h
    A[N] = 2 + h**2*q(x)
    C[N] = -1 - (h/2)*p(x)
    D[N] = -h**2*r(x) + (1 - (h/2)*p(x))*beta

    # now tridiagonal, and crout:
    l[1] = A[1]
    u[1] = B[1]/l[1]
    z[1] = D[1]/l[1]

    for i in range(2,N):
        l[i] = A[i] - C[i]*u[i-1]
        u[i] = B[i]/l[i]
        z[i] = (D[i] - C[i]*z[i-1])/l[i]
    
    l[N] = A[N] - C[N]*u[N-1]
    z[N] = (D[N] - C[N]*z[N-1])/l[N]

    # now the solution
    w[0] = alpha
    w[N+1] = beta
    w[N] = z[N]
    for i in range(N-1, 0, -1):
        w[i] = z[i] - u[i]*w[i+1]

    # print the final values
    xs = np.array([a + i*h for i in range(N+2)])
    return xs, w

a = np.array(linfin(p, q, r, 0, np.pi/4, 0, 0, 4))
print("The estimates from this method are \n", a)

# now calculate the difference between the est and the true soln:
def soln2(x):
    return (-1/3)*np.cos(2*x) - (np.sqrt(2)/6)*np.sin(2*x) + 1/3*np.cos(x)

true2 = np.array(soln2(np.linspace(0, np.pi/4, 6)))
error = true2 - a[1]
print("At each step, the errors are: \n", error)


The estimates from this method are 
 [[ 0.          0.15707963  0.31415927  0.4712389   0.62831853  0.78539816]
 [ 0.         -0.06141845 -0.09240491 -0.09080499 -0.05825827  0.        ]]
At each step, the errors are: 
 [ 0.00000000e+00  7.93053804e-04  1.20910327e-03  1.19161248e-03
  7.58768356e-04 -5.55111512e-17]
